In [ ]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7 triton==2.0.0 bitsandbytes==0.40.2 huggingface_hub datasets

In [ ]:
from sklearn.model_selection import train_test_split
from huggingface_hub import login, HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig, EarlyStoppingCallback
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, PeftModel, PeftConfig
from trl import SFTTrainer
from datasets import Dataset, DatasetDict
import json
import os, gc, torch
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
# Đăng nhập vào Hugging Face
login()  # Đảm bảo bạn đã login với token phù hợp

In [ ]:
# Thêm vào cell import
torch.backends.cuda.matmul.allow_tf32 = True  # Dòng mới cần thêm
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB")

In [ ]:
# Các tham số và thông tin người dùng
hf_username = "thanhdat2004"  # Thay bằng username của bạn trên Hugging Face
current_chunk = 5            # Thay đổi số này cho mỗi session mới (1, 2, 3, 4, 5)
previous_chunk = current_chunk - 1  # Chunk trước đó

In [ ]:
# Định nghĩa model, tokenizer và các tham số huấn luyện
base_model_name = 'vilm/vinallama-7b-chat'  # Model gốc
previous_model_path = f"{hf_username}/MealCaloCalculator_vinallama_chunk{previous_chunk}" if previous_chunk > 0 else None

In [ ]:
# Cấu hình tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name if previous_chunk == 0 else previous_model_path)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# Cấu hình lượng tử hóa
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
# Tải model gốc
print(f"🔄 Tải model gốc: {base_model_name}")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

In [ ]:
# Chuẩn bị model cho huấn luyện
base_model = prepare_model_for_kbit_training(base_model)

In [ ]:
# Cấu hình PEFT (LoRA)
peft_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    fan_in_fan_out=True
)

In [ ]:
# Cấu hình huấn luyện
training_args = TrainingArguments(
    output_dir=f"trained_model_chunk{current_chunk}",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-5,
    optim="adamw_torch_fused",
    max_grad_norm=0.8, # giảm từ 1
    warmup_ratio=0.1,
    lr_scheduler_type="cosine", # chuyển từ linear
    fp16 = True, # fp16=False,
    bf16=False,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    gradient_checkpointing=True,
    remove_unused_columns=False,
    evaluation_strategy="steps",
    eval_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

In [ ]:
# Thêm callback cho early stopping
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.005
)
callbacks = [early_stopping_callback]

In [ ]:
# Tải dữ liệu huấn luyện
print(f"📊 Đang tải dữ liệu huấn luyện từ chunk {current_chunk}")
data_files = [
    f"/kaggle/input/meal-dataset/split_{current_chunk}.jsonl"
]

In [ ]:
# Xử lý dữ liệu
transformed_data = []
for file in data_files:
    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            transformed_data.append(json.loads(line))

In [ ]:
# Chia dữ liệu train/val
train_data, val_data = train_test_split(transformed_data, test_size=0.1)
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

print(f"📈 Tổng số mẫu dữ liệu: {len(transformed_data)}")
print(f"🧪 Số mẫu huấn luyện: {len(train_data)}")
print(f"🔍 Số mẫu đánh giá: {len(val_data)}")

In [ ]:
# Chuẩn bị các thông số cho repository
chunk_model_name = f"MealCaloCalculator_vinallama_chunk{current_chunk}"
repo_name = f"{hf_username}/{chunk_model_name}"
adapter_dir = f"./adapter_chunk_{current_chunk}"


In [ ]:
# Cấu hình phân mảnh bộ nhớ
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

# Khởi tạo biến theo dõi
best_loss = float("inf")
no_improve_count = 0
patience = 2

In [ ]:
# Kiểm tra và tải model từ chunk trước nếu cần
if previous_chunk > 0:
    print(f"🔄 Tải adapter từ chunk trước: {previous_model_path}")
    model = PeftModel.from_pretrained(
        base_model,
        previous_model_path,
        device_map="auto",
        torch_dtype=torch.float16,
        is_trainable=True
    )
else:
    print("🔄 Sử dụng model gốc cho lần finetune đầu tiên")
    model = base_model

In [ ]:
print(f"Train dataset: {train_dataset}")
print(f"Val dataset: {val_dataset}")
print(f"PEFT Config: {peft_config}")
print(f"Training args: {training_args}")

In [ ]:
# Huấn luyện model
print(f"\n🚀 Bắt đầu huấn luyện chunk {current_chunk}")
print(f"💻 GPU Memory: Allocated = {torch.cuda.memory_allocated()/1024**2:.2f}MB, Reserved = {torch.cuda.memory_reserved()/1024**2:.2f}MB")

try:
    # Tạo trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        peft_config=peft_config,
        dataset_text_field="text",
        max_seq_length=64,
        args=training_args,
        packing=False,
        callbacks=callbacks
    )
    
    # Huấn luyện
    trainer.train()
    
    # Đánh giá
    with torch.inference_mode():
        eval_results = trainer.evaluate()
        current_loss = eval_results["eval_loss"]
        print(f"📉 Evaluation Loss: {current_loss:.4f}")
        
        # Lưu model
        print(f"💾 Lưu model tốt nhất vào {adapter_dir}")
        # trainer.model.save_pretrained(adapter_dir)
        # tokenizer.save_pretrained(adapter_dir)
        
        # # Tạo card mô tả cho model
        # model_card = f"""
        # # MealCaloCalculator VinaLLaMA - Chunk {current_chunk}
        
        # Model này là một phiên bản finetune của [vilm/vinallama-7b-chat](https://huggingface.co/vilm/vinallama-7b-chat) trên dữ liệu dinh dưỡng, chunk {current_chunk}.
        
        # ## Training
        # - Dữ liệu: Meal dataset (chunk {current_chunk})
        # - Phương pháp: PEFT với LoRA
        # - Eval Loss: {current_loss:.4f}
        # """
        
        # with open(f"{adapter_dir}/README.md", "w", encoding="utf-8") as f:
        #     f.write(model_card)
        # Thay thế đoạn code hiện tại bằng:
        trainer.model.save_pretrained(adapter_dir, safe_serialization=True)
        tokenizer.save_pretrained(adapter_dir)
        
        # Tự viết file README.md với encoding utf-8
        model_card = f"""---
language:
- vi
- en
license: apache-2.0
---

# MealCaloCalculator VinaLLaMA - Chunk {current_chunk}

## Training Details
- **Data**: Meal dataset (chunk {current_chunk})
- **Method**: LoRA (r=4, alpha=8)
- **Eval Loss**: {current_loss:.4f}
- **Hardware**: Kaggle (T4 GPU)
"""
        with open(f"{adapter_dir}/README.md", "w", encoding="utf-8") as f:
            f.write(model_card)
        
        # Push model lên Hugging Face Hub
        print(f"🚀 Pushing model {chunk_model_name} to Hugging Face Hub...")
        trainer.model.push_to_hub(
            repo_id=repo_name,
            use_auth_token=True,
            commit_message=f"Training chunk {current_chunk} với eval_loss: {current_loss:.4f}"
        )
        
        # Đẩy tokenizer lên cùng với model
        tokenizer.push_to_hub(
            repo_id=repo_name,
            use_auth_token=True
        )
        
        print(f"✅ Model {chunk_model_name} đã được push lên Hugging Face Hub!")

except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("⚠️ OOM Error! Thực hiện cleanup...")
        torch.cuda.empty_cache()
        gc.collect()
    raise
    
finally:
    # Giải phóng bộ nhớ triệt để
    del trainer
    torch.cuda.empty_cache()
    gc.collect()
    print(f"🧹 Memory after cleanup: {torch.cuda.memory_allocated()/1024**2:.2f}MB")

In [ ]:
# Tải model đã huấn luyện để kiểm tra
print("\n🔄 Loading model vừa finetune từ Hugging Face Hub...")

# Load tokenizer và model
tokenizer = AutoTokenizer.from_pretrained(repo_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

print(f"🔍 Loading adapter từ {repo_name}...")
peft_model = PeftModel.from_pretrained(
    base_model,
    repo_name,
    device_map="auto",
    torch_dtype=torch.float16,
    is_trainable=False  # Chế độ inference
)

In [ ]:
# Hàm generate để sử dụng model
def generate_answer(query):
    prompt = f"""<|im_start|>system
Bạn là chuyên gia dinh dưỡng AI. Hãy đưa ra DUY NHẤT MỘT thực đơn phù hợp<|im_end|>
<|im_start|>user
{query}<|im_end|>
<|im_start|>assistant
Thực đơn gợi ý:
"""
    
    # Tokenize và chuyển ĐÚNG device với model
    inputs = tokenizer(prompt, return_tensors="pt").to(peft_model.device)
    
    # Generate
    with torch.inference_mode():
        outputs = peft_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_k=40,
            top_p=0.9,
            repetition_penalty=1.5
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Kiểm tra model và tokenizer
print(f"🔍 Model device: {next(peft_model.parameters()).device}")
print(f"🔍 Model dtype: {next(peft_model.parameters()).dtype}")
print(f"🔍 Tokenizer pad token: {tokenizer.pad_token}")



In [ ]:
# Test với câu hỏi ngắn
test_query = "Hãy đề xuất thực đơn có tổng 2400 calo trong 1 ngày."
try:
    print("\n📝 Thử nghiệm model với câu hỏi:")
    print(f"🔤 Query: {test_query}")
    response = generate_answer(test_query)
    print(f"💬 Response: {response}")
    print("\n✅ Model hoạt động tốt!")
except Exception as e:
    print(f"❌ Lỗi: {e}")